<a href="https://colab.research.google.com/github/reza-abdolee/Generative-AI-Systems---Winter-2027/blob/main/Module_03_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 03 Lab: Model Selection, Performance, and Cost Tradeoffs
**COMP 490 Generative AI Systems · CSU Channel Islands · Dr. Reza Abdolee**

Run the cells **from top to bottom, in order**. Your NRP API key must be saved in Colab Secrets (🔑) as `NRP_API_KEY` with notebook access turned on (see Module 00).

| Tier | Role | Model | Backup |
|---|---|---|---|
| Tier 1 (Frontier) | The Logical Core | `glm-5` | `qwen3` |
| Tier 1 (Agentic) | The Multi-Step Brain | `kimi` | `qwen3` |
| Tier 2 (Performance) | The Production Standard | `gpt-oss` | `gemma` |
| Tier 2 (Specialist) | The Instruction Specialist | `qwen3-small` | `gemma` |
| Tier 3 (Edge) | The Speed Scout | `gemma-small` | `gemma` |
| Tier 3 (Functional) | The Tool-User | `qwen3` + your tool | `gpt-oss` + your tool |

`glm-5`, `kimi` and `gemma-small` are still in testing on the NRP. If one is unavailable, the notebook uses its backup automatically and records which model actually answered.

## Setup
### Step 1: Install libraries and connect to the NRP

In [ ]:
%pip install -q -U openai pandas matplotlib yfinance

import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI, BadRequestError
from google.colab import userdata

# Secure API initialization: the key is read from Colab Secrets, never typed here.
client = OpenAI(
    api_key=userdata.get("NRP_API_KEY").strip(),
    base_url="https://ellm.nrp-nautilus.io/v1",
)

### Step 2: The tier matrix
Each role has a primary model and a stable backup. `effort` is the thinking setting (`reasoning_effort`); each value is one that model accepts. `None` means the model has no adjustable levels, so no setting is sent.

In [ ]:
TIERS = {
    "Tier 1 - Frontier":    {"role": "The Logical Core",           "model": "glm-5",       "effort": "high",
                             "backup": "qwen3",   "backup_effort": "medium"},
    "Tier 1 - Agentic":     {"role": "The Multi-Step Brain",       "model": "kimi",        "effort": None,
                             "backup": "qwen3",   "backup_effort": "medium"},
    "Tier 2 - Performance": {"role": "The Production Standard",    "model": "gpt-oss",     "effort": "low",
                             "backup": "gemma",   "backup_effort": "none"},
    "Tier 2 - Specialist":  {"role": "The Instruction Specialist", "model": "qwen3-small", "effort": "low",
                             "backup": "gemma",   "backup_effort": "none"},
    "Tier 3 - Scout":       {"role": "The Speed Scout",            "model": "gemma-small", "effort": "none",
                             "backup": "gemma",   "backup_effort": "none"},
    "Tier 3 - Tool":        {"role": "The Tool-User",              "model": "qwen3",       "effort": "low",
                             "backup": "gpt-oss", "backup_effort": "low"},
}
pd.DataFrame(TIERS).T

### Step 3: Helper functions
`ask()` sends one request, measures its duration, and retries if the shared NRP service is briefly busy. `ask_tier()` tries the tier's primary model and falls back to its backup if the primary is unavailable.

In [ ]:
def ask(model, messages, effort=None, temperature=None, max_tokens=4000, tools=None, retries=2):
    """Send one chat request. Returns (response, duration_seconds)."""
    kwargs = {"model": model, "messages": messages, "max_tokens": max_tokens}
    if temperature is not None:
        kwargs["temperature"] = temperature
    if effort is not None:
        kwargs["extra_body"] = {"reasoning_effort": effort}
    if tools is not None:
        kwargs["tools"] = tools
    for attempt in range(1, retries + 1):
        try:
            start = time.time()
            response = client.chat.completions.create(**kwargs)
            return response, time.time() - start
        except BadRequestError:
            raise                      # HTTP 400: a setting is wrong, retrying will not help
        except Exception as error:
            if attempt == retries:
                raise
            print(f"  {model}: attempt {attempt} failed ({error}); retrying in 10 s")
            time.sleep(10)


def ask_tier(tier, messages, **kwargs):
    """Try the tier's primary model, then its backup. Returns (response, duration, model_used, effort_used)."""
    cfg = TIERS[tier]
    for model, effort in [(cfg["model"], cfg["effort"]), (cfg["backup"], cfg["backup_effort"])]:
        try:
            response, duration = ask(model, messages, effort=effort, **kwargs)
            return response, duration, model, effort
        except Exception as error:
            print(f"  {tier}: {model} failed ({error}); trying backup")
    raise RuntimeError(f"{tier}: both the primary and backup models failed")

### Connection check
Confirms which of this lab's models are available right now.

In [ ]:
available = [m.id for m in client.models.list().data]
for tier, cfg in TIERS.items():
    primary = "OK" if cfg["model"] in available else "NOT AVAILABLE (backup will be used)"
    print(f"{tier:22s} {cfg['model']:12s} {primary}")

## Part 3: Decoding & Stochasticity
**Model:** `gemma` with thinking turned off, so only the temperature changes between runs.

**Prompt:** "Tell me a 3-sentence story about a banana in space."
- **Test A:** three runs at temperature 0.0
- **Test B:** three runs at temperature 1.8

In [ ]:
DECODING_MODEL, DECODING_EFFORT = "gemma", "none"
decoding_prompt = "Tell me a 3-sentence story about a banana in space."
decoding_rows = []

for label, temp in [("Test A - Deterministic", 0.0), ("Test B - Stochastic", 1.8)]:
    for run in range(1, 4):
        response, duration = ask(DECODING_MODEL, [{"role": "user", "content": decoding_prompt}],
                                 effort=DECODING_EFFORT, temperature=temp, max_tokens=400)
        text = response.choices[0].message.content or ""
        decoding_rows.append({"Test_Label": label, "Model": DECODING_MODEL, "Temperature": temp,
                              "Run_Number": run, "Generated_Text": text})
        print(f"[{label} | run {run}]\n{text}\n")
        time.sleep(2)   # share the NRP service fairly

decoding_df = pd.DataFrame(decoding_rows)
decoding_df.to_csv("decoding_results.csv", index=False)
print("Saved decoding_results.csv")

In [ ]:
# Were the three temperature 0.0 outputs identical?
for label, group in decoding_df.groupby("Test_Label"):
    unique = group["Generated_Text"].nunique()
    print(f"{label}: {unique} unique output(s) out of {len(group)} runs")

## Part 4: Reasoning vs. Speed Tournament
The same logic puzzle goes to all six tiers. The expected answer is **4 cats**.

**Note:** duration is the total request time measured with `time.time()`, and TPS is `completion_tokens / duration`. This does not use streaming, so it does not measure Time to First Token. Completion tokens include hidden thinking, so thinking models can show high TPS while the visible answer is short.

In [ ]:
cats_prompt = ("A room has 4 corners. A cat is in every corner. In front of each cat are 3 cats. "
               "How many cats are in the room total? Explain your step-by-step logic.")
tournament_rows = []

for tier, cfg in TIERS.items():
    print(f"Running {tier} ...")
    try:
        response, duration, model_used, effort_used = ask_tier(tier, [{"role": "user", "content": cats_prompt}])
    except Exception as error:
        print(f"  skipped: {error}")
        continue
    usage = response.usage
    tournament_rows.append({
        "Tier": tier, "Role": cfg["role"], "Model": model_used,
        "Used Backup": model_used != cfg["model"], "Thinking": effort_used or "built-in (fixed)",
        "Completion Tokens": usage.completion_tokens, "Total Tokens": usage.total_tokens,
        "Duration (sec)": round(duration, 2),
        "TPS": round(usage.completion_tokens / duration, 1) if duration > 0 else None,
        "Response": response.choices[0].message.content or "",
    })
    time.sleep(2)

tournament_df = pd.DataFrame(tournament_rows)
tournament_df.to_csv("tournament_results.csv", index=False)
print("Saved tournament_results.csv")
tournament_df.drop(columns="Response")

**Read each answer** and decide whether it reached 4 cats with sound reasoning.

In [ ]:
for _, row in tournament_df.iterrows():
    print(f"===== {row['Tier']} | {row['Model']} =====")
    print(row["Response"][-1200:], "\n")   # the conclusion is usually at the end

In [ ]:
# Label each model yourself: "Correct" or "Incorrect" (the expected answer is 4).
CORRECT = {tier: "" for tier in tournament_df["Tier"]}
CORRECT   # edit the values above, e.g. CORRECT["Tier 1 - Frontier"] = "Correct"

### Throughput chart

In [ ]:
labels = [f"{t}\n({m})" for t, m in zip(tournament_df["Tier"], tournament_df["Model"])]
plt.figure(figsize=(11, 5))
plt.bar(labels, tournament_df["TPS"], color="#2E74B5")
plt.ylabel("Approximate tokens per second (TPS)")
plt.title("Reasoning vs. Speed Tournament: Throughput by Tier")
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.tight_layout()
plt.savefig("tps_chart.png", dpi=150)
plt.show()
print("Saved tps_chart.png")

## Part 5: Real-Time Tool Integration
**Static model:** the Tier 3 Scout answers from its training data only.

**Tool-using model:** the Tier 3 Tool-User can call `get_stock_price()`, a Python function you provide. The model decides whether to call it; your code runs the function and sends the result back; the model then writes the final answer.

**Prompt:** "What is the current stock price of NVIDIA (NVDA)?"

In [ ]:
def get_stock_price(ticker):
    """Return the latest available (possibly delayed) price for a U.S. stock ticker."""
    ticker = ticker.upper().strip()
    try:   # Source 1: Stooq free quote service (no API key)
        quote = pd.read_csv(f"https://stooq.com/q/l/?s={ticker.lower()}.us&f=sd2t2ohlcv&h&e=csv")
        price = float(quote.loc[0, "Close"])
        return {"ticker": ticker, "price_usd": price, "date": str(quote.loc[0, "Date"]),
                "time": str(quote.loc[0, "Time"]), "source": "Stooq (delayed quote)"}
    except Exception:
        pass
    try:   # Source 2: Yahoo Finance through the yfinance library
        import yfinance as yf
        price = float(yf.Ticker(ticker).fast_info["last_price"])
        return {"ticker": ticker, "price_usd": round(price, 2), "source": "Yahoo Finance via yfinance"}
    except Exception as error:
        return {"ticker": ticker, "error": f"Price lookup failed: {error}"}

# Test the tool by itself before giving it to a model.
get_stock_price("NVDA")

In [ ]:
# The tool description the model sees (JSON schema).
STOCK_TOOL = [{
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description": "Get the latest available stock price for a U.S. ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {"ticker": {"type": "string", "description": "Ticker symbol, e.g. NVDA"}},
            "required": ["ticker"],
        },
    },
}]
TOOL_FUNCTIONS = {"get_stock_price": get_stock_price}

In [ ]:
stock_prompt = "What is the current stock price of NVIDIA (NVDA)?"
tool_rows = []

# 1) Static model: no tools available.
response, duration, model_used, _ = ask_tier("Tier 3 - Scout", [{"role": "user", "content": stock_prompt}])
tool_rows.append({"Model_Type": "Static", "Model_ID": model_used, "Tool_Called": False,
                  "Tool_Arguments": "", "Tool_Result": "",
                  "Response_Content": response.choices[0].message.content or ""})
time.sleep(2)

# 2) Tool-using model: first call offers the tool.
messages = [{"role": "user", "content": stock_prompt}]
response, duration, model_used, effort_used = ask_tier("Tier 3 - Tool", messages, tools=STOCK_TOOL)
message = response.choices[0].message
tool_called, tool_args, tool_result = False, "", ""

if message.tool_calls:
    tool_called = True
    # Record the model's tool request in the conversation.
    messages.append({"role": "assistant", "content": message.content or "",
                     "tool_calls": [{"id": tc.id, "type": "function",
                                     "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                                    for tc in message.tool_calls]})
    # Run each requested function and send the result back.
    for tc in message.tool_calls:
        args = json.loads(tc.function.arguments or "{}")
        result = TOOL_FUNCTIONS[tc.function.name](**args)
        tool_args, tool_result = json.dumps(args), json.dumps(result)
        print(f"Model called {tc.function.name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": tool_result})
    # Second call: the model writes the final answer using the tool result.
    response, _ = ask(model_used, messages, effort=effort_used, tools=STOCK_TOOL)
    message = response.choices[0].message
else:
    print("The model answered without calling the tool.")

tool_rows.append({"Model_Type": "Tool-Using", "Model_ID": model_used, "Tool_Called": tool_called,
                  "Tool_Arguments": tool_args, "Tool_Result": tool_result,
                  "Response_Content": message.content or ""})

tool_df = pd.DataFrame(tool_rows)
tool_df.to_csv("tool_results.csv", index=False)
for _, row in tool_df.iterrows():
    print(f"\n===== {row['Model_Type']} ({row['Model_ID']}) | tool called: {row['Tool_Called']} =====")
    print(row["Response_Content"])
print("\nSaved tool_results.csv")

## Part 6: Output Length, Cost Control, and Efficiency Scaling
**Model:** `qwen3-small` with thinking turned off, so completion tokens count only the visible answer.
- **Experiment A (Unconstrained):** "Explain the concept of Cloud Computing in detail."
- **Experiment B (Budgeted):** the same user prompt plus a system instruction limiting the answer to 2 sentences and fewer than 50 tokens.

In [ ]:
BUDGET_MODEL, BUDGET_EFFORT = "qwen3-small", "none"
user_prompt = "Explain the concept of Cloud Computing in detail."
system_prompt = ("You are a concise technical writer. Explain Cloud Computing in exactly 2 sentences, "
                 "using fewer than 50 tokens.")

experiments = {
    "Unconstrained": [{"role": "user", "content": user_prompt}],
    "Budgeted": [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
}
budget_rows = []
for name, msgs in experiments.items():
    response, duration = ask(BUDGET_MODEL, msgs, effort=BUDGET_EFFORT, max_tokens=4000)
    budget_rows.append({"Experiment": name, "Model": BUDGET_MODEL, "Thinking": BUDGET_EFFORT,
                        "Completion Tokens": response.usage.completion_tokens,
                        "Total Tokens": response.usage.total_tokens,
                        "Duration (sec)": round(duration, 2),
                        "Response": response.choices[0].message.content or ""})
    time.sleep(2)

budget_df = pd.DataFrame(budget_rows)
unconstrained = budget_df.loc[budget_df["Experiment"] == "Unconstrained", "Completion Tokens"].iloc[0]
budget_df["Efficiency Gain %"] = ((unconstrained - budget_df["Completion Tokens"]) / unconstrained * 100).round(1)
budget_df.to_csv("budgeting_results.csv", index=False)
print("Budgeted answer:\n", budget_df.loc[1, "Response"])
budget_df.drop(columns="Response")

**Optional Experiment C:** run the budgeted prompt again with thinking turned on (`low`). Compare its completion tokens with Experiment B: the visible answer is the same length, but hidden thinking is billed too.

In [ ]:
response, duration = ask(BUDGET_MODEL, experiments["Budgeted"], effort="low", max_tokens=4000)
print("Thinking on (low): completion tokens =", response.usage.completion_tokens)
print("Thinking off      : completion tokens =", budget_df.loc[1, "Completion Tokens"])
print("Answer:", response.choices[0].message.content)

## Files to submit
Download these from the folder icon on the left (⋮ → Download).

In [ ]:
import os
for f in ["decoding_results.csv", "tournament_results.csv", "tool_results.csv", "budgeting_results.csv", "tps_chart.png"]:
    print(f"{'OK     ' if os.path.exists(f) else 'MISSING'} {f}")

## Before you submit
- [ ] All cells ran from top to bottom.
- [ ] The `CORRECT` labels in Part 4 are filled in.
- [ ] The five files above are saved (`decoding_results.csv`, `tournament_results.csv`, `tool_results.csv`, `budgeting_results.csv`, `tps_chart.png`).
- [ ] This notebook does not contain your API key.
- [ ] Submit this notebook (Share link or .ipynb) with your Technical Report.